# Exercise: Does the Loss Mask Matter?

Built from `BabyQwen-2-FineTuning.ipynb`. We change **one thing**, the loss mask, and keep data, model, LoRA settings and seed fixed.

- **Run A**: response-only loss (prompt labels = -100)
- **Run B**: full-sequence loss (prompt included)
- **Stretch**: Run A with 250 examples

**Hypothesis:** training on the prompt wastes capacity on text the model is given at inference, so masking should follow instructions better.


---
## 1. Setup (original Cells 1–8)

In [11]:
import subprocess, sys
try:
    import peft
except ImportError:   # e.g. a fresh Colab runtime
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'peft'])

import os, re, math, time, random
from collections import Counter
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import load_dataset
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
random.seed(0)
torch.manual_seed(0)
print(f'PyTorch {torch.__version__} | device: {device.upper()}')

PyTorch 2.11.0+cu130 | device: CUDA


### Find the Week 4 checkpoint (Google Drive)

In [12]:
import glob, os, sys

IN_COLAB = 'google.colab' in sys.modules
DRIVE_ROOT = '/content/drive/MyDrive/ADS-NLP/BabyQwen'   # same folder the Week 4 save cell writes to
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

SEARCH = [os.path.join(DRIVE_ROOT, 'baby_qwen_pretrained'), 'baby_qwen_pretrained', '/content/baby_qwen_pretrained',
          '../week4/baby_qwen_pretrained', '../Week4/baby_qwen_pretrained']

def find_ckpt():
    for p in SEARCH:
        if os.path.isfile(os.path.join(p, 'config.json')):
            return p
    # Search Google Drive a few levels deep
    for depth in range(0, 5):
        hits = glob.glob('/content/drive/MyDrive/' + '*/' * depth + 'baby_qwen_pretrained/config.json')
        if hits:
            return os.path.dirname(hits[0])
    return None

FOUND_DIR = find_ckpt()
if FOUND_DIR is None and IN_COLAB:
    from google.colab import files
    print('Not found. Upload baby_qwen_pretrained.zip:')
    for name in files.upload():
        if name.endswith('.zip'):
            os.system(f'unzip -o -q "{name}" -d /content')
    FOUND_DIR = find_ckpt()

assert FOUND_DIR, 'baby_qwen_pretrained not found. Run Week 4 cells 1-16 on Colab first (cell 16 copies it to Drive).'
print('Using checkpoint:', FOUND_DIR, '|', sorted(os.listdir(FOUND_DIR)))

RESULTS_DIR = os.path.join(DRIVE_ROOT if IN_COLAB else '.', 'loss_mask_results')
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Results will be saved to:', RESULTS_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using checkpoint: /content/drive/MyDrive/ADS-NLP/BabyQwen/baby_qwen_pretrained | ['chat_template.jinja', 'config.json', 'generation_config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json']
Results will be saved to: /content/drive/MyDrive/ADS-NLP/BabyQwen/loss_mask_results


In [13]:
# Output folder of BabyQwen-1. On Colab, upload it next to this notebook (or into /content) first.
CANDIDATES = [FOUND_DIR, '../week4/baby_qwen_pretrained', 'baby_qwen_pretrained', '/content/baby_qwen_pretrained']
BASE_DIR = next((p for p in CANDIDATES if os.path.isdir(p)), None)
assert BASE_DIR, f'baby_qwen_pretrained not found in {CANDIDATES}. Run week4/BabyQwen-1-Pretraining.ipynb first.'

tokenizer = AutoTokenizer.from_pretrained(BASE_DIR)
IM_START = tokenizer.convert_tokens_to_ids('<|im_start|>')
IM_END = tokenizer.convert_tokens_to_ids('<|im_end|>')
EOT = tokenizer.convert_tokens_to_ids('<|endoftext|>')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_base():
    return AutoModelForCausalLM.from_pretrained(BASE_DIR, dtype=torch.float32).to(device)

model = load_base()
n_params = sum(p.numel() for p in model.parameters())
print(f'Loaded Baby Qwen2.5: {n_params:,} parameters')
print(f'Special token ids: <|im_start|>={IM_START}  <|im_end|>={IM_END}  <|endoftext|>={EOT}')

Loading weights:   0%|          | 0/50 [00:00<?, ?it/s]

Loaded Baby Qwen2.5: 41,844,736 parameters
Special token ids: <|im_start|>=151644  <|im_end|>=151645  <|endoftext|>=151643


> **A subtle point.** `<|im_start|>` and `<|im_end|>` exist in the Qwen vocabulary, but Baby Qwen never saw them in pretraining. Its stories were delimited with `<|endoftext|>`. The embeddings for the chat tokens are therefore still at their random initialization. A real pretrained Qwen has learned them. SFT must teach Baby Qwen both what the template *means* and when to emit `<|im_end|>` to stop.

In [14]:
SYSTEM = 'You are a friendly storyteller.'

def chat_prompt(instruction):
    return (f'<|im_start|>system\n{SYSTEM}<|im_end|>\n'
            f'<|im_start|>user\n{instruction}<|im_end|>\n'
            f'<|im_start|>assistant\n')

MASK_PROMPT = True   # EXERCISE: switch for Run B

def encode_example(ex):
    # Returns token ids and labels. If MASK_PROMPT, prompt tokens get label -100 (ignored by the loss).
    prompt_ids = tokenizer.encode(chat_prompt(ex['instr']))
    resp_ids = tokenizer.encode(ex['resp']) + [IM_END]
    ids = prompt_ids + resp_ids
    if MASK_PROMPT:
        return ids, [-100] * len(prompt_ids) + resp_ids
    return ids, ids[:]            # loss on every token, prompt included

# Show the masking on a tiny example
demo = {'instr': 'Write a short story about a character named Mia.', 'resp': 'Mia had a red ball.'}
ids, labels = encode_example(demo)
print(f"{'token':<16} {'trained on?'}")
for i, l in list(zip(ids, labels))[-9:]:
    print(f'{tokenizer.decode([i])!r:<16} {"YES" if l != -100 else "no (masked)"}')

token            trained on?
'\n'             no (masked)
'M'              YES
'ia'             YES
' had'           YES
' a'             YES
' red'           YES
' ball'          YES
'.'              YES
'<|im_end|>'     YES


In [15]:
# ── USE YOUR OWN DATA (optional) ───────────────────────────────────────────
# CUSTOM_PAIRS: a list of {'instr': ..., 'resp': ...} dicts, or the path to a .jsonl file with
#               one such object per line. Leave as None to use the TinyStories tasks below.
# The ChatML template, system prompt and response-only loss are unchanged. A custom run has no
# automatic word/name check, so watch the held-out response loss and read the generated samples.
CUSTOM_PAIRS = None
CUSTOM_VAL_FRACTION = 0.1

PRETRAIN_STORIES = 6_000   # used by BabyQwen-1, so we skip them
N_POOL = 12_000            # candidate stories for SFT
N_TRAIN = 1_000      # EXERCISE: default 4,000
N_VAL_STORIES = 400
MAX_LEN = 192   # short stories only: faster steps, and the instruction is never truncated away

def take(split, skip, n):
    if CUSTOM_PAIRS is not None: return []   # no need to download TinyStories
    ds = load_dataset('roneneldan/TinyStories', split=split, streaming=True).skip(skip)
    return [ex['text'].strip() for _, ex in zip(range(n), ds)]

pool = take('train', PRETRAIN_STORIES, N_POOL)
val_stories = take('validation', 0, N_VAL_STORIES)

# Document frequency over the pool: content words are neither too rare nor too common
doc_freq = Counter(w for s in pool for w in set(re.findall(r'[a-z]{4,}', s.lower())))
def is_content_word(w):
    return 10 <= doc_freq[w] <= 0.04 * len(pool)

NOT_NAMES = {'I', 'Mom', 'Dad', 'Mommy', 'Daddy', 'Grandma', 'Grandpa', 'Mama', 'Papa'}
def sentences(story):
    return re.split(r'(?<=[.!?])\s+', story)

def build_words(story, rng):
    cands = sorted(w for w in set(re.findall(r'[a-z]{4,}', story.lower())) if is_content_word(w))
    if len(cands) < 1: return None
    ws = rng.sample(cands, 1)
    return dict(kind='words', instr=f'Write a short story using the word: {ws[0]}.', resp=story, meta=ws)

def build_name(story, rng):
    # Names are capitalized words that appear mid-sentence at least twice
    counts = Counter()
    for sent in sentences(story):
        for tok in sent.split()[1:]:
            tok = tok.strip('.,!?"\'')
            tok = tok[:-2] if tok.endswith("'s") else tok
            if tok.istitle() and tok.isalpha() and tok not in NOT_NAMES:
                counts[tok] += 1
    if not counts or counts.most_common(1)[0][1] < 2: return None
    name = counts.most_common(1)[0][0]
    return dict(kind='name', instr=f'Write a short story about a character named {name}.', resp=story, meta=name)

def build_continue(story, rng):
    s = sentences(story)
    if len(s) < 5: return None
    return dict(kind='continue', instr='Continue this story: ' + ' '.join(s[:2]), resp=' '.join(s[2:]), meta=None)

BUILDERS = [build_words, build_name, build_continue]

def make_examples(stories, n, seed):
    # Rotate through the three types, falling back to the next type if a story doesn't fit
    rng = random.Random(seed)
    out = []
    for i, story in enumerate(stories):
        for k in range(3):
            ex = BUILDERS[(i + k) % 3](story, rng)
            if ex is not None:
                ids, _ = encode_example(ex)
                if len(ids) <= MAX_LEN:
                    out.append(ex)
                break
        if len(out) >= n: break
    return out

train_examples = make_examples(pool, N_TRAIN, seed=1)
val_examples = make_examples(val_stories, 10_000, seed=2)

if CUSTOM_PAIRS is not None:
    if isinstance(CUSTOM_PAIRS, str):
        import json
        CUSTOM_PAIRS = [json.loads(line) for line in open(CUSTOM_PAIRS, encoding='utf-8') if line.strip()]
    bad = [k for k, p in enumerate(CUSTOM_PAIRS)
           if not (isinstance(p, dict) and str(p.get('instr', '')).strip() and str(p.get('resp', '')).strip())]
    assert not bad, f"Pairs at indexes {bad[:5]} need non-empty 'instr' and 'resp' fields."
    pairs = [dict(kind='custom', instr=p['instr'].strip(), resp=p['resp'].strip(), meta=None) for p in CUSTOM_PAIRS]
    usable = [p for p in pairs if len(encode_example(p)[0]) <= MAX_LEN]   # drop examples longer than MAX_LEN tokens
    print(f'{len(usable)} of {len(pairs)} pairs fit within MAX_LEN={MAX_LEN} tokens')
    assert len(usable) >= 20, 'Need at least 20 usable pairs (raise MAX_LEN if your examples are long).'
    random.Random(0).shuffle(usable)
    n_val = max(2, int(CUSTOM_VAL_FRACTION * len(usable)))
    val_examples, train_examples = usable[:n_val], usable[n_val:]
print(f'Train examples: {len(train_examples)} | Val examples: {len(val_examples)}')
print('Train mix:', dict(Counter(e['kind'] for e in train_examples)))
for kind in ['words', 'name', 'continue', 'custom']:
    ex = next((e for e in train_examples if e['kind'] == kind), None)
    if ex:
        print(f"\n[{kind}] {ex['instr'][:110]}\n   -> {ex['resp'][:130]}...")

Train examples: 1000 | Val examples: 154
Train mix: {'continue': 542, 'words': 291, 'name': 167}

[words] Write a short story using the word: chicken.
   -> Once upon a time, there was a big farm. On the farm, there was a large chicken. The chicken was very happy and liked to walk aroun...

[name] Write a short story about a character named Boney.
   -> Once there was a dog. The dog had a bone. The bone was very big. The dog loved the bone. One day, the dog lost the bone. The dog w...

[continue] Continue this story: Once upon a time, there was a big red truck. It was very loud and made lots of noise.
   -> One day, the truck was driving down the road and saw a little girl playing with her toys. The truck was very nosy and wanted to se...


### Training and evaluation utilities (original Section 4)

In [16]:
def collate(examples):
    # Right-pad a batch; padded label positions are -100 so they are ignored
    enc = [encode_example(e) for e in examples]
    L = max(len(i) for i, _ in enc)
    ids = torch.full((len(enc), L), tokenizer.pad_token_id, dtype=torch.long)
    lab = torch.full((len(enc), L), -100, dtype=torch.long)
    att = torch.zeros((len(enc), L), dtype=torch.long)
    for r, (i, l) in enumerate(enc):
        ids[r, :len(i)] = torch.tensor(i); lab[r, :len(l)] = torch.tensor(l); att[r, :len(i)] = 1
    return ids.to(device), lab.to(device), att.to(device)

@torch.no_grad()
def response_loss(m, examples, bs=8):
    # Token-weighted mean cross-entropy over response tokens only
    m.eval()
    total, count = 0.0, 0
    for i in range(0, len(examples), bs):
        ids, lab, att = collate(examples[i:i + bs])
        logits = m(input_ids=ids, attention_mask=att).logits[:, :-1]
        tgt = lab[:, 1:]
        total += F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), tgt.reshape(-1),
                                 ignore_index=-100, reduction='sum').item()
        count += (tgt != -100).sum().item()
    return total / count

def train(m, data, val_data, epochs, lr, bs=8, warmup_frac=0.05, label=''):
    params = [p for p in m.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, betas=(0.9, 0.95), weight_decay=0.0)
    steps_per_epoch = math.ceil(len(data) / bs)
    total_steps = epochs * steps_per_epoch
    warmup = max(1, int(warmup_frac * total_steps))
    def lr_at(s):
        if s < warmup: return lr * (s + 1) / warmup
        p = (s - warmup) / max(1, total_steps - warmup)
        return 0.1 * lr + 0.9 * lr * 0.5 * (1 + math.cos(math.pi * p))

    log = {'step': [], 'loss': [], 'val_step': [0], 'val_loss': [response_loss(m, val_data)]}
    print(f'[{label}] step 0 | val loss {log["val_loss"][0]:.4f}')
    t0, step = time.time(), 0
    for ep in range(epochs):
        order = list(range(len(data))); random.Random(ep).shuffle(order)
        m.train()
        for b in range(0, len(order), bs):
            ids, lab, att = collate([data[j] for j in order[b:b + bs]])
            for g in opt.param_groups: g['lr'] = lr_at(step)
            logits = m(input_ids=ids, attention_mask=att).logits[:, :-1]
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), lab[:, 1:].reshape(-1), ignore_index=-100)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step(); step += 1
            log['step'].append(step); log['loss'].append(loss.item())
        v = response_loss(m, val_data)
        log['val_step'].append(step); log['val_loss'].append(v)
        print(f'[{label}] epoch {ep + 1}/{epochs} | step {step}/{total_steps} | train loss {log["loss"][-1]:.4f} | val loss {v:.4f} | {time.time() - t0:.0f}s')
    return log, time.time() - t0

@torch.no_grad()
def generate(m, instructions, max_new_tokens=220, bs=20):
    # Batched greedy decoding with left padding; stops at <|im_end|> or <|endoftext|>
    m.eval()
    tokenizer.padding_side = 'left'
    outs = []
    for i in range(0, len(instructions), bs):
        enc = tokenizer([chat_prompt(x) for x in instructions[i:i + bs]], return_tensors='pt', padding=True).to(m.device)
        g = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1,
                       eos_token_id=[IM_END, EOT], pad_token_id=tokenizer.pad_token_id)
        outs += tokenizer.batch_decode(g[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
    tokenizer.padding_side = 'right'
    return [o.strip() for o in outs]

def follows(ex, text):
    if ex['kind'] == 'words':
        return all(re.search(rf'\b{w}\b', text.lower()) for w in ex['meta'])
    return re.search(rf'\b{ex["meta"]}\b', text) is not None

N_EVAL = 60
eval_words = [e for e in val_examples if e['kind'] == 'words'][:N_EVAL]
eval_names = [e for e in val_examples if e['kind'] == 'name'][:N_EVAL]
loss_val = val_examples[:150]   # mixed-type held-out set for response loss
DEMO_PROMPTS = [
    'Write a short story using the word: kite.',
    'Write a short story about a character named Mia.',
    'Continue this story: One day, a little fox found a shiny key. He wanted to know what it opened.',
]
if CUSTOM_PAIRS is not None:
    DEMO_PROMPTS = [e['instr'] for e in val_examples[:3]]   # held-out prompts from your data

def evaluate(m, label):
    # Instruction-following rate for the two checkable task types, plus held-out response loss
    res = {'label': label, 'val_loss': response_loss(m, loss_val)}
    for kind, exs in [('words', eval_words), ('name', eval_names)]:
        outs = generate(m, [e['instr'] for e in exs]) if exs else []
        res[kind] = sum(follows(e, o) for e, o in zip(exs, outs)) / len(exs) if exs else float('nan')
    res['demo'] = generate(m, DEMO_PROMPTS)
    print(f"[{label}] val loss {res['val_loss']:.3f} | words {res['words']:.0%} | name {res['name']:.0%}")
    return res

print(f'Eval prompts: {len(eval_words)} words, {len(eval_names)} name | loss set: {len(loss_val)}')

Eval prompts: 47 words, 33 name | loss set: 150


### Extra helpers for the exercise
- `stop_rate`: share of generations that emit `<|im_end|>` before hitting `max_new_tokens`. (`generate()` strips special tokens, so we check the raw ids.)
- `fresh_lora`: same base, same `LoraConfig`, same seed, so the only difference between runs is the mask.

In [17]:
EPOCHS = 1      # EXERCISE: default 3
BS = 16
LR = 1e-3
SEED = 0

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

def fresh_lora():
    # Same seed every run, so LoRA init and dropout match across runs
    random.seed(SEED); torch.manual_seed(SEED)
    return get_peft_model(load_base(), lora_config)

STOP_PROMPTS = [e['instr'] for e in (eval_words[:20] + eval_names[:20])]

@torch.no_grad()
def stop_rate(m, instructions=STOP_PROMPTS, max_new_tokens=220, bs=20):
    # Fraction of greedy generations that emit <|im_end|> (and how many end with <|endoftext|> instead)
    m.eval()
    tokenizer.padding_side = 'left'
    n_im, n_eot = 0, 0
    for i in range(0, len(instructions), bs):
        enc = tokenizer([chat_prompt(x) for x in instructions[i:i + bs]], return_tensors='pt', padding=True).to(m.device)
        g = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1,
                       eos_token_id=[IM_END, EOT], pad_token_id=tokenizer.pad_token_id)
        for row in g[:, enc['input_ids'].shape[1]:].tolist():
            n_im += IM_END in row
            n_eot += (EOT in row) and (IM_END not in row)
    tokenizer.padding_side = 'right'
    return n_im / len(instructions), n_eot / len(instructions)

def evaluate_full(m, label, n_train, mask):
    # Always score held-out loss on response tokens only, the same way for every run
    global MASK_PROMPT
    MASK_PROMPT = True
    res = evaluate(m, label)
    res['stop_im_end'], res['stop_eot'] = stop_rate(m)
    res.update(mask=mask, n_train=n_train)
    print(f"[{label}] stops at <|im_end|>: {res['stop_im_end']:.0%} | ends at <|endoftext|> instead: {res['stop_eot']:.0%}")
    return res

import json
RESULTS_FILE = os.path.join(RESULTS_DIR, 'results.json')

def save_results():
    with open(RESULTS_FILE, 'w') as f:
        json.dump(results, f, indent=1)
    print(f'Saved results ({", ".join(results)}) to {RESULTS_FILE}')

def load_results():
    # RESUME = True: runs already in results.json are skipped (useful after a Colab disconnect)
    if RESUME and os.path.isfile(RESULTS_FILE):
        r = json.load(open(RESULTS_FILE))
        print(f'Resuming: found finished runs {list(r)} in {RESULTS_FILE}')
        return r
    return {}

RESUME = True   # set False to redo every run from scratch

def free(m):
    del m
    if device == 'mps': torch.mps.empty_cache()
    if device == 'cuda': torch.cuda.empty_cache()

---
## 2. Base model ("before SFT" row)

In [18]:
results = load_results()
if 'base' not in results:
    results['base'] = evaluate_full(model, 'Base (no SFT)', n_train=0, mask='n/a')
    save_results()
free(model)

Resuming: found finished runs ['base', 'lora'] in /content/drive/MyDrive/ADS-NLP/BabyQwen/loss_mask_results/results.json


---
## 3. Run A: response-only loss (baseline)

In [10]:
import subprocess, sys
try:
    import torchao
    # Check if torchao needs an upgrade
    from lora_torchao import TORCHAO_MINIMUM_VERSION
except ImportError:
    pass

# Upgrade torchao to meet the minimum version requirement for PEFT
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'torchao>=0.16.0'])

if 'lora' not in results:
    MASK_PROMPT = True
    lora_model = fresh_lora()
    lora_model.print_trainable_parameters()
    lora_log, lora_time = train(lora_model, train_examples, loss_val, epochs=EPOCHS, lr=LR, bs=BS, label='LoRA, masked')
    results['lora'] = evaluate_full(lora_model, 'LoRA, masked', n_train=len(train_examples), mask='response only')
    results['lora']['time'] = lora_time
    save_results()   # metrics only. Do not overwrite baby_qwen_lora_adapter/ or baby_qwen_sft/.
    free(lora_model)

Loading weights:   0%|          | 0/50 [00:00<?, ?it/s]

trainable params: 303,104 || all params: 42,147,840 || trainable%: 0.7191
[LoRA, masked] step 0 | val loss 3.5460
[LoRA, masked] epoch 1/1 | step 63/63 | train loss 3.7632 | val loss 3.3581 | 12s
[LoRA, masked] val loss 3.358 | words 0% | name 39%
[LoRA, masked] stops at <|im_end|>: 0% | ends at <|endoftext|> instead: 8%
Saved results (base, lora) to /content/drive/MyDrive/ADS-NLP/BabyQwen/loss_mask_results/results.json


---
## 4. Run B: full-sequence loss (no masking)

The val loss printed *during* this run includes prompt tokens, so it is not comparable. Use the `evaluate` numbers (scored with `MASK_PROMPT = True`).

In [19]:
if 'nomask' not in results:
    MASK_PROMPT = False
    lora_nomask = fresh_lora()
    nomask_log, nomask_time = train(lora_nomask, train_examples, loss_val, epochs=EPOCHS, lr=LR, bs=BS, label='LoRA, no mask')

    MASK_PROMPT = True
    results['nomask'] = evaluate_full(lora_nomask, 'LoRA, no mask', n_train=len(train_examples), mask='full sequence')
    results['nomask']['time'] = nomask_time
    save_results()
    free(lora_nomask)

Loading weights:   0%|          | 0/50 [00:00<?, ?it/s]

[LoRA, no mask] step 0 | val loss 4.5489
[LoRA, no mask] epoch 1/1 | step 63/63 | train loss 3.8736 | val loss 3.4855 | 12s
[LoRA, no mask] val loss 3.393 | words 0% | name 39%
[LoRA, no mask] stops at <|im_end|>: 0% | ends at <|endoftext|> instead: 0%
Saved results (base, lora, nomask) to /content/drive/MyDrive/ADS-NLP/BabyQwen/loss_mask_results/results.json


### Mechanism: what fraction of Run B's loss tokens come from the prompt?

In [20]:
MASK_PROMPT = True
n_prompt = n_resp = 0
by_kind = Counter(); tot_kind = Counter()
for ex in train_examples:
    p = len(tokenizer.encode(chat_prompt(ex['instr'])))
    r = len(tokenizer.encode(ex['resp'])) + 1      # + <|im_end|>
    n_prompt += p; n_resp += r
    by_kind[ex['kind']] += p; tot_kind[ex['kind']] += p + r
# Each position predicts the next token, so a sequence of length L has L-1 targets; the first prompt token is never a target
n_targets = n_prompt + n_resp - len(train_examples)
prompt_frac = (n_prompt - len(train_examples)) / n_targets
print(f'Run B loss tokens: {n_targets:,} | from prompt: {n_prompt - len(train_examples):,} ({prompt_frac:.1%}) | from response: {n_resp:,} ({n_resp / n_targets:.1%})')
for k in tot_kind:
    print(f'  {k:<9} prompt share of tokens: {by_kind[k] / tot_kind[k]:.1%}')
sys_len = len(tokenizer.encode(f'<|im_start|>system\n{SYSTEM}<|im_end|>\n'))
print(f'System prompt alone: {sys_len} tokens, identical in all {len(train_examples):,} examples')

Run B loss tokens: 168,301 | from prompt: 38,267 (22.7%) | from response: 130,034 (77.3%)
  continue  prompt share of tokens: 28.2%
  words     prompt share of tokens: 17.5%
  name      prompt share of tokens: 17.4%
System prompt alone: 12 tokens, identical in all 1,000 examples


---
## 5. Stretch: Run A with 250 examples

`make_examples` is deterministic, so the first 250 of `train_examples` are exactly what `N_TRAIN = 250` would give.

In [21]:
RUN_STRETCH = True
if RUN_STRETCH and 'small' not in results:
    MASK_PROMPT = True
    small_train = train_examples[:250]
    lora_small = fresh_lora()
    small_log, small_time = train(lora_small, small_train, loss_val, epochs=EPOCHS, lr=LR, bs=BS, label='LoRA, masked, 250')
    results['small'] = evaluate_full(lora_small, 'LoRA, masked, 250', n_train=len(small_train), mask='response only')
    results['small']['time'] = small_time
    save_results()
    free(lora_small)

Loading weights:   0%|          | 0/50 [00:00<?, ?it/s]

[LoRA, masked, 250] step 0 | val loss 3.5460
[LoRA, masked, 250] epoch 1/1 | step 16/16 | train loss 3.9073 | val loss 3.4072 | 4s
[LoRA, masked, 250] val loss 3.407 | words 0% | name 61%
[LoRA, masked, 250] stops at <|im_end|>: 0% | ends at <|endoftext|> instead: 25%
Saved results (base, lora, nomask, small) to /content/drive/MyDrive/ADS-NLP/BabyQwen/loss_mask_results/results.json


---
## 6. Results Table

In [22]:
def ci95(p, n):
    return 1.96 * math.sqrt(p * (1 - p) / n)

ROWS = [('base', 'Base (no SFT)'), ('lora', 'Run A'), ('nomask', 'Run B'), ('small', 'Stretch')]
ROWS = [(k, n) for k, n in ROWS if k in results]

hdr = f"| Run | Loss mask | Train examples | Held-out response loss | Words follow-rate | Name follow-rate | Stops at <\\|im_end\\|>? |"
lines = [hdr, '|' + '---|' * 7]
for k, name in ROWS:
    r = results[k]
    lines.append(f"| {name} | {r['mask']} | {r['n_train']:,} | {r['val_loss']:.3f} | "
                 f"{r['words']:.0%} ± {ci95(r['words'], len(eval_words)):.0%} | "
                 f"{r['name']:.0%} ± {ci95(r['name'], len(eval_names)):.0%} | "
                 f"{r['stop_im_end']:.0%} ({len(STOP_PROMPTS)} prompts) |")
table_md = '\n'.join(lines)

try:
    from IPython.display import Markdown, display
    display(Markdown(table_md))
except ImportError:
    pass
print(table_md)   # copy-paste into your write-up
with open(os.path.join(RESULTS_DIR, 'results_table.md'), 'w') as f:
    f.write(table_md + '\n')
print('Table saved to', os.path.join(RESULTS_DIR, 'results_table.md'))

| Run | Loss mask | Train examples | Held-out response loss | Words follow-rate | Name follow-rate | Stops at <\|im_end\|>? |
|---|---|---|---|---|---|---|
| Base (no SFT) | n/a | 0 | 3.546 | 0% ± 0% | 55% ± 17% | 0% (40 prompts) |
| Run A | response only | 1,000 | 3.358 | 0% ± 0% | 39% ± 17% | 0% (40 prompts) |
| Run B | full sequence | 1,000 | 3.393 | 0% ± 0% | 39% ± 17% | 0% (40 prompts) |
| Stretch | response only | 250 | 3.407 | 0% ± 0% | 61% ± 17% | 0% (40 prompts) |

| Run | Loss mask | Train examples | Held-out response loss | Words follow-rate | Name follow-rate | Stops at <\|im_end\|>? |
|---|---|---|---|---|---|---|
| Base (no SFT) | n/a | 0 | 3.546 | 0% ± 0% | 55% ± 17% | 0% (40 prompts) |
| Run A | response only | 1,000 | 3.358 | 0% ± 0% | 39% ± 17% | 0% (40 prompts) |
| Run B | full sequence | 1,000 | 3.393 | 0% ± 0% | 39% ± 17% | 0% (40 prompts) |
| Stretch | response only | 250 | 3.407 | 0% ± 0% | 61% ± 17% | 0% (40 prompts) |
Table saved to /content/drive/MyDrive/ADS-NLP/BabyQwen/loss_mask_results/results_table.md


### Is the A vs B gap outside sampling noise? (60 prompts per metric)

In [23]:
def compare(a, b, la, lb):
    print(f'--- {la} vs {lb} ---')
    print(f"val loss: {results[a]['val_loss']:.3f} vs {results[b]['val_loss']:.3f} (diff {results[b]['val_loss'] - results[a]['val_loss']:+.3f})")
    for key, n in (('words', len(eval_words)), ('name', len(eval_names))):
        x, y = results[a][key], results[b][key]
        noise = math.sqrt(ci95(x, n) ** 2 + ci95(y, n) ** 2)
        print(f'{key:<6} {x:.0%} vs {y:.0%} | diff {y - x:+.0%} | noise ±{noise:.0%} | outside noise: {abs(y - x) > noise}')

compare('lora', 'nomask', 'Run A (masked)', 'Run B (no mask)')
if 'small' in results:
    compare('base', 'lora', 'Base', 'Run A (1,000)')
    compare('base', 'small', 'Base', 'Stretch (250)')
    for key in ('words', 'name'):
        gap_full = results['lora'][key] - results['base'][key]
        gap_small = results['small'][key] - results['base'][key]
        if gap_full > 0:
            print(f'{key}: {gap_small / gap_full:.0%} of the base→masked gain survives with 4x less data')

--- Run A (masked) vs Run B (no mask) ---
val loss: 3.358 vs 3.393 (diff +0.035)
words  0% vs 0% | diff +0% | noise ±0% | outside noise: False
name   39% vs 39% | diff +0% | noise ±24% | outside noise: False
--- Base vs Run A (1,000) ---
val loss: 3.546 vs 3.358 (diff -0.188)
words  0% vs 0% | diff +0% | noise ±0% | outside noise: False
name   55% vs 39% | diff -15% | noise ±24% | outside noise: False
--- Base vs Stretch (250) ---
val loss: 3.546 vs 3.407 (diff -0.139)
words  0% vs 0% | diff +0% | noise ±0% | outside noise: False
name   55% vs 61% | diff +6% | noise ±24% | outside noise: False


---
## 7. Demo generations (one per run)

For each, note: does it **repeat the prompt**, **ramble past the end**, or **stop cleanly**?

In [24]:
DEMO_IDX = 1   # 0 = kite, 1 = Mia, 2 = continue

def flags(text, prompt):
    t = text.lower()
    echo = any(s in t for s in ('<|im_start|>', 'system', 'user', 'assistant', 'friendly storyteller', 'write a short story'))
    echo = echo or prompt.lower()[:40] in t
    return f"echoes template/prompt: {echo} | words: {len(text.split())}"

print('PROMPT:', DEMO_PROMPTS[DEMO_IDX])
for k, name in ROWS:
    out = results[k]['demo'][DEMO_IDX]
    print('=' * 78)
    print(f'{name}  ({results[k]["mask"]}, {results[k]["n_train"]:,} ex) -> {flags(out, DEMO_PROMPTS[DEMO_IDX])}')
    print(out[:500])

# Also check whether the demo stopped at <|im_end|> or hit the token limit
print('\n' + '=' * 78)
print('Stop check on the 3 demo prompts is in the table column; per-run stop rates:')
for k, name in ROWS:
    print(f"  {name:<14} <|im_end|> {results[k]['stop_im_end']:.0%} | <|endoftext|> {results[k]['stop_eot']:.0%} | "
          f"hit max tokens {1 - results[k]['stop_im_end'] - results[k]['stop_eot']:.0%}")

PROMPT: Write a short story about a character named Mia.
Base (no SFT)  (n/a, 0 ex) -> echoes template/prompt: False | words: 30
, and her mom were very happy.

The moral of the best of the day, they were playing together. They had a lot of fun and had a great time.
Run A  (response only, 1,000 ex) -> echoes template/prompt: False | words: 171
Once upon a time, there was a little girl named Lily. She loved to play with her friends. One day, she saw a big tree in the park. She wanted to play with it, but she was too late. She asked her mom if she could help her mom and her mom said, "I'm sorry, I can't be careful." 

Lily smiled and said, "I want to play with me?" Lily said, "I'm sorry, Lily. I will be careful with you." Lily was happy and said, "I'm sorry, Lily. You can't find it."

Lily was happy and said, "Let's go to the park, Lily
Run B  (full sequence, 1,000 ex) -> echoes template/prompt: False | words: 185
Once upon a time, there was a little girl named Lily. She loved to play wi

### All three demo prompts, side by side (optional)

In [25]:
for i, prompt in enumerate(DEMO_PROMPTS):
    print('=' * 78)
    print('PROMPT:', prompt)
    for k, name in ROWS:
        print(f'\n--- {name} ---')
        print(results[k]['demo'][i][:420])

PROMPT: Write a short story using the word: kite.

--- Base (no SFT) ---
, it was so happy and he had been seen. He was so excited to be careful and he could not find it. 

The little girl was so happy that she had never seen something special. She was so happy and she had to be careful.

--- Run A ---
Once upon a time, there was a little girl named Lily. She loved to play with her friends. One day, she saw a big tree in the park. She wanted to play with it, but she was too late. She asked her mom if she could help her mom and her mom said, "I'm sorry, I can't be careful." 

Lily smiled and said, "I want to play with me?" 

Lily was so happy that she had a special place to make her new things. She was very happy an

--- Run B ---
Once upon a time, there was a little girl named Lily. She loved to play with her friends. One day, she saw a big, and said, "I'm sorry, I can't worry." 

Lily was so happy that she had to be careful and always wanted to help her. She asked her mom if she could

---
## 8. Results and Analysis

### Results table

| Run | Loss mask | Train examples | Held-out response loss | Words follow-rate | Name follow-rate | Stops at <\|im_end\|>? |
|---|---|---|---|---|---|---|
| Base (no SFT) | n/a | 0 | 3.546 | 0% | 55% ± 17% | No: 0% (ends at `<\|endoftext\|>` 95%) |
| Run A | response only | 1,000 | **3.358** | 0% | 39% ± 17% | No: 0% (hits token limit 92%) |
| Run B | full sequence | 1,000 | 3.393 | 0% | 39% ± 17% | No: 0% (hits token limit 100%) |
| Stretch | response only | 250 | 3.407 | 0% | 61% ± 17% | No: 0% (hits token limit 75%) |

Eval set sizes: 47 words prompts, 33 name prompts (not 60: the held-out pool only had 154 examples). Stop rate is measured on 40 prompts. All runs: LoRA r=16, lr=1e-3, 1 epoch, seed 0.

### Demo generations (prompt: "Write a short story about a character named Mia.")

| Run | Repeats prompt? | Rambles past the end? | Stops cleanly? | Note |
|---|---|---|---|---|
| Base | No | No | Ends at `<\|endoftext\|>`, not `<\|im_end\|>` | 30 words. Starts mid-sentence (", and her mom were very happy"). Not a chat answer. |
| Run A | No | Yes | No, cut at the token limit | 171 words. A "Lily" story, ignores "Mia". Same story for the kite prompt. Mild looping ("I'm sorry"). |
| Run B | No | Yes, worst | No, cut at the token limit | 185 words. Same "Lily" story, broken grammar ("she saw a big, and saw a big"), ends in a loop ("They were happy and they had a fun day" x3). |
| Stretch | No | Yes | No, cut at the token limit | 116 words. "The little girl", no name at all. Loops on "She was so happy and...". |

### Analysis

**Result.** Masking won slightly on held-out response loss: Run A 3.358 vs Run B 3.393 (both down from 3.546 for the base model). On follow-rates there is no difference: both runs got 0% on words and 39% on names. With 33 name prompts, each rate is ±17 points and the A vs B gap would need to beat about ±24 points, so the follow-rates cannot separate the two runs. The 39% is also misleading: both models write the same default "Lily" story for almost every prompt, and about 40% of the held-out name prompts ask for "Lily" (most common TinyStories name), so most of those "hits" are luck. The hypothesis is supported by the loss, but not shown by the behaviour metrics at this training size.

**Mechanism.** In Run B, 22.7% of the loss tokens come from the prompt (38,267 of 168,301). It is 28% for "continue" examples and about 17% for words and name examples. Part of this is the 12-token system prompt, which is identical in all 1,000 examples, so the model learns it fast and the gradient on it is wasted. The rest is the user instruction, which the model is always given at inference, so learning to predict it does not help it answer. It also pulls the small LoRA adapter (303k parameters) toward modelling the template instead of the mapping from instruction to response, which is the only thing we want it to learn.

**Behavior.** Neither model echoed the template or the instruction in the demos (the echo flag is False for every run). The visible difference is quality: Run B has more broken grammar and falls into a repeated loop sooner than Run A. A model trained on the prompt learns that text after `<|im_start|>` looks like system and user turns, so with more training it can start writing "user" or "Write a short story..." lines itself, because that text was a target it was rewarded for predicting. With only 63 steps this did not show up yet; it shows up as noisier output instead.

**The stop token.** No run learned to emit `<|im_end|>`: the rate is 0% for all four. The base model stops only because it emits `<|endoftext|>` (95%) after a short, off-task text. Run B never stops at all (100% hit the 220-token limit), Run A almost never does (92%). The reason is that `<|im_end|>` has a random, untrained embedding, and LoRA freezes the embedding table; the embeddings are also tied to the output layer, so its output row is frozen too, and 63 steps were not enough to learn it through the adapters alone. Without the stop token, generation only ends at `max_new_tokens`, so the model rambles and loops, wastes compute, and in a real chat it would run into a fake next turn. It also makes the follow-rate less fair, since longer text has more chances to contain the target word.

**Handoff.** BabyQwen-3 loads `baby_qwen_sft/` with `AutoModelForCausalLM.from_pretrained`. That is the full fine-tuned model from the original notebook (3 epochs, 4,000 examples), not a LoRA adapter. A Run A adapter folder only has `adapter_config.json` and `adapter_model.safetensors`, so it would not even load there without merging it into the base first. Even if merged, it is a much weaker starting point: it ignores the instruction, does not stop at `<|im_end|>`, and was trained on 4x less data for 1/3 of the epochs. BabyQwen-3 measures fact recall and forgetting, so starting from this model would mix "did not learn the fact" with "never learned to answer in the chat format," and its numbers would drop for reasons that have nothing to do with knowledge injection.

**Stretch (4x less data).** With 250 examples the held-out loss is 3.407. Run A cut the loss by 0.188 from base, the stretch run cut it by 0.139, so about 74% of the loss gain survives with 4x less data. The follow-rates (0% words, 61% names) are within noise of the base model, so they show no gain to compare.
